# Universidad Central · Big Data · Sesión 13
# ¿Cuándo basta pandas y cuándo necesitamos distribuir datos y modelos?

**Caso continuo:** Laura / Compras Claras. **Entorno principal:** Codespaces — elegir perfil **S13 · Dask, MapReduce y modelos SECOP**.

[Presentación y tutorial S13](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) · [Datos de referencia](../Datos/tc2_corpus_2026.json)

**No dependemos de que S12 ya se haya impartido.** El Taller 2 es una actividad evaluativa con su propia fecha. Aquí utilizamos dos fuentes versionadas:
- **200 expedientes** del Taller 2 para continuidad narrativa, *sin etiqueta de modalidad*.
- **1.000 registros SECOP** para entrenar un clasificador didáctico de modalidad, *no de fraude*.

Al final obtendrás datos Parquet, un manifiesto de origen, un clasificador evaluado y una explicación sobre cuándo Dask agrega valor. Todos los registros base son del repositorio; no necesitas APIs ni cuentas de pago.


## Mapa del laboratorio · recorrido principal y práctica opcional

1. Elegir herramienta: pandas, SQL/DuckDB, Dask, Spark.
2. Entender archivos distribuidos, bloques y réplicas sin instalar Hadoop.
3. Resolver MapReduce manualmente, luego en Python.
4. Entrar a Codespaces y comprobar bibliotecas.
5. Comparar pandas con Dask sobre el **mismo archivo SECOP**.
6. Entrenar **un modelo de scikit-learn** sobre modalidad.
7. Coordinar **varios entrenamientos independientes** con Dask Distributed.
8. Registrar un fallo controlado, guardar Parquet y adaptar el método.

**Convención:** después de cada salida encontrarás: **Cómo se lee / Qué nos dice / Qué NO permite concluir / Error común.** Los tiempos están sólo en el libreto docente, no aquí.


## 0 · Decidir antes de instalar

**Pregunta profesional:** ¿qué problema justifica pagar el costo de coordinar trabajadores?

| Trabajo | Primera opción razonable |
|---|---|
| 200 registros que caben en memoria | pandas |
| Consulta SQL sobre muchos CSV/Parquet | considerar DuckDB |
| Muchas transformaciones independientes o datos mayores que memoria | Dask puede ayudar |
| Un modelo clásico con cientos de ejemplos | scikit-learn |
| Un único modelo profundo entrenado entre máquinas | algoritmo con soporte distribuido explícito |

**OJO:** Dask no es sinónimo de «más rápido». Dos workers dentro de un Codespace **no son dos computadores**. Cada herramienta resuelve una necesidad distinta.


## 1 · Crear tu entorno propio en el navegador

**HAZ ESTO AHORA.** 1. Abre [el repositorio](https://github.com/jazaineam1/BigData2026). 2. Pulsa **Code → Codespaces → ⋯ → New with options**. 3. En configuración del contenedor selecciona **S13 · Dask, MapReduce y modelos SECOP**. **No** elijas «Airflow Lab» (predeterminado) ni «S11 multimedia». 4. Crea un Codespace y abre una **Terminal → New Terminal**.

Dentro de la terminal ejecuta:

```bash
python --version
python -m pip show pandas
python -m pip show dask
python -m pip show scikit-learn
python Laboratorios/S13_Codespaces/verificar_entorno.py
```

Si falta una biblioteca, **dentro de Codespaces**, no en tu PC: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`. Abre este cuaderno en VS Code web y selecciona el kernel Python del Codespace. No necesita Ubuntu, Docker ni VS Code instalados localmente.

**Resultados:** debe terminar con `LISTO PARA S13`; si escogiste el perfil equivocado, crea otro Codespace con la configuración S13.


### 1A · Preparar rutas (por qué no importa si abriste el cuaderno desde Cuadernos)

`Path` maneja rutas; `sys.path` indica dónde encontrar las funciones reutilizables. Conserva visible el código: puedes adaptarlo a un proyecto ajeno.


In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "Datos").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "Datos/tc2_corpus_2026.json").exists(), "Abre el repositorio en Codespaces"
sys.path.insert(0, str(RAIZ / "Laboratorios/S13_Codespaces"))
print("Raíz del curso:", RAIZ)


### 1B · Comprobar biblioteca y archivos, no sólo que pip haya respondido

Esta celda ejecuta un cálculo mínimo Dask y comprueba que existen las dos fuentes del curso.


In [ ]:
import pandas as pd
import dask.dataframe as dd
from analisis import leer_corpus, leer_etiquetados

demo = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
mini = dd.from_pandas(demo, npartitions=2)
print(mini.groupby("grupo")["valor"].sum().compute())
assert int(mini.groupby("grupo")["valor"].sum().compute().loc["A"]) == 30
print("Fuentes y herramientas:", len(leer_corpus()), len(leer_etiquetados()))


**Cómo se lee:** A suma 30; B suma 30. Se encuentran 200 expedientes de referencia y 1.000 procesos etiquetados.  
**Qué nos dice:** pandas y Dask están disponibles y pueden ejecutar un cálculo.  
**Qué NO permite concluir:** que haya un clúster físico o mayor rendimiento que pandas.  
**Error común:** asumir que «pip instaló» equivale a «el flujo completo funciona».


## 2 · Sistemas de archivos: bloques, nodos y réplicas

Un sistema como **HDFS** divide archivos en bloques lógicos y almacena esos bloques en nodos. La replicación crea copias para tolerar fallos. El **NameNode** administra metadatos de ubicación; los **DataNodes** guardan bloques.

Esto NO es lo mismo que partir un DataFrame para procesarlo. La siguiente celda es una **SIMULACIÓN en Python**, no un clúster Hadoop real.

| Elemento | Función |
|---|---|
| Nodo | máquina o proceso que participa |
| Bloque | unidad de almacenamiento |
| Réplica | otra copia de un mismo bloque |
| Partición | subconjunto lógico de trabajo |
| Scheduler | coordinador de tareas |
| Worker | ejecutor de tareas |

**Decisión:** ¿qué sucede si se pierde un nodo que guarda la única copia de un bloque?


In [ ]:
ubicaciones = {
    "bloque_1": ["nodo_A", "nodo_B"],
    "bloque_2": ["nodo_B", "nodo_C"],
    "bloque_3": ["nodo_A", "nodo_C"],
}
nodo_caido = "nodo_B"

for bloque, nodos in ubicaciones.items():
    disponibles = [n for n in nodos if n != nodo_caido]
    print(bloque, "→", disponibles)
    assert disponibles, f"Bloque no recuperable: {bloque}"


**Cómo se lee:** cada bloque conserva una ubicación distinta de nodo_B.  
**Qué nos dice:** copiar bloques en distintos nodos puede proteger contra una falla simple.  
**Qué NO permite concluir:** que un Codespace sea un HDFS real ni que todas las políticas de replicación garanticen disponibilidad.  
**Error común:** confundir replicar los datos con paralelizar el cálculo.


## 3 · MapReduce: la trampa del promedio de promedios

Imagina 3 lotes de contrataciones con diferentes tamaños. En el primer lote hay 2 importes, en el segundo 5 y en el tercero 100. **Promediar las tres medias de los lotes es incorrecto** porque no representan el mismo número de observaciones.

`map` produce estadísticas parciales. `reduce` combina parciales sin perder los pesos. **Es el modelo MapReduce implementado en Python**, no un servicio de Apache Hadoop.


In [ ]:
lotes = [
    [100, 120],
    [10, 12, 11, 9, 8],
    [1] * 100,
]
parciales = [(sum(lote), len(lote)) for lote in lotes]
print("MAP (suma, cantidad):", parciales)


In [ ]:
suma_total = sum(suma for suma, cantidad in parciales)
cantidad_total = sum(cantidad for suma, cantidad in parciales)

promedio_correcto = suma_total / cantidad_total
promedio_incorrecto = sum(s / n for s, n in parciales) / len(parciales)

print("REDUCE correcto:", round(promedio_correcto, 3))
print("Promedio de promedios:", round(promedio_incorrecto, 3))
assert round(promedio_correcto, 6) == round(370/107, 6)


**Cómo se lee:** 370 unidades en 107 observaciones dan una media de aproximadamente 3,458.  
**Qué nos dice:** para un promedio distribuido necesitamos **suma y cantidad**, no sólo medias locales.  
**Qué NO permite concluir:** que ya usamos Hadoop MapReduce ni que un algoritmo cualquiera sea divisible así.  
**Error común:** `mean(mean(lote))` ignora los tamaños de las particiones.

**Transfiere:** esta estrategia sirve para métricas de ventas, llamadas o tiempos por cliente, siempre que conserves los pesos.


## 4 · ¿Qué trae el caso SECOP de Laura?

La S12 es el **Taller de Control 2**, no una clase de Dask terminada por definición. Su corpus de 200 expedientes no incluye la modalidad contractual como etiqueta para un modelo supervisado. Lo usamos sólo para recuperar la historia y comprobar identidad.


In [ ]:
referencia = leer_corpus()
print("Expedientes del taller:", len(referencia))
print(referencia[["id_proceso", "entidad"]].head(3))
assert referencia["id_proceso"].is_unique


### 4A · Datos para el problema supervisado

El CSV de SECOP versionado sí tiene `descripci_n_del_procedimiento` y `modalidad_de_contratacion`.

**Pregunta del modelo:** ¿podemos aprender a clasificar una modalidad registrada a partir del texto? **No es un detector de corrupción**.


In [ ]:
tabla = leer_etiquetados()
print("Dimensiones:", tabla.shape)
print(tabla["modalidad_de_contratacion"].value_counts().head(7))
print(tabla[["id_del_proceso", "descripci_n_del_procedimiento"]].head(2))
assert tabla["id_del_proceso"].is_unique


**Cómo se lee:** 1.000 filas, 59 columnas; «Contratación directa» y «Contratación régimen especial» son clases frecuentes.  
**Qué nos dice:** hay etiquetas registradas y texto para plantear una clasificación didáctica.  
**Qué NO permite concluir:** que esas frecuencias representen todo SECOP ni que el texto contenga evidencia de fraude.  
**Error común:** entrenar sobre los 200 expedientes del Taller 2 como si tuvieran esa etiqueta.


## 5 · pandas primero: la referencia de corrección

`groupby` agrupa filas por modalidad y `size()` cuenta filas. Esta salida es la referencia contra la cual compararemos Dask.


In [ ]:
conteo_pandas = tabla.groupby("modalidad_de_contratacion").size().sort_index()
print(conteo_pandas.head(10))
print("Total:", int(conteo_pandas.sum()))
assert conteo_pandas.sum() == 1000


## 6 · Dask DataFrame: el mismo cálculo por particiones

`from_pandas` divide el DataFrame en particiones. `groupby` construye operaciones diferidas; `compute()` materializa el resultado. Una partición **no** es un servidor.

¿Cuál sería el costo de Dask en sólo mil registros? El coordinador, el grafo de tareas y la comunicación pueden tardar más que la operación real.


In [ ]:
dask_tabla = dd.from_pandas(tabla, npartitions=4)
pendiente = dask_tabla.groupby("modalidad_de_contratacion").size()

print("Particiones:", dask_tabla.npartitions)
print("Antes de compute:", type(pendiente).__name__)
conteo_dask = pendiente.compute().sort_index()
valores_pandas = {str(k): int(v) for k, v in conteo_pandas.items()}
valores_dask = {str(k): int(v) for k, v in conteo_dask.items()}
assert valores_pandas == valores_dask
print("Resultado Dask:", conteo_dask.head(), "\nCoincide: SÍ")


### 6A · Medir sin inventar «Dask siempre gana»

Una medición debe incluir el tiempo de ejecución, el número de particiones y el tamaño del conjunto. Se repite en condiciones comparables; el primer tiempo puede incluir inicialización, cachés y procesos. Comparamos por **corrección** antes de velocidad.


In [ ]:
from analisis import comparar_pandas_dask
mediciones = []
for n_particiones in (1, 2, 4):
    mediciones.append(comparar_pandas_dask(tabla, npartitions=n_particiones))

display(pd.DataFrame(mediciones)[
    ["filas", "particiones", "segundos_pandas", "segundos_dask", "coinciden"]
])


**Cómo se lee:** los segundos se miden en TU Codespace y pueden cambiar en cada ejecución.  
**Qué nos dice:** la comparación es local, sobre el mismo conjunto de 1.000 registros.  
**Qué NO permite concluir:** cuál herramienta es más rápida para millones de filas en otro sistema.  
**Error común:** confundir aumentar particiones con aumentar máquinas o comparar ejecuciones con cargas distintas.


## 7 · Entrenamiento de modelos: ¿qué significa `fit()`?

**Entrenamiento**: el modelo ajusta parámetros usando ejemplos etiquetados. **Inferencia**: utiliza esos parámetros para clasificar nuevos textos.

Aquí `X` es una descripción de SECOP y `y` es su modalidad registrada. **TF-IDF** transforma palabras en variables numéricas. La regresión logística clasifica. La división entrenamiento/prueba se realiza **antes** de ajustar TF-IDF para evitar fuga de información estadística.

**Nunca entrenamos un modelo de «fraude» ni confundimos la etiqueta modalidad con irregularidad.**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from analisis import preparar_clasificacion

muestra = preparar_clasificacion(tabla)
X = muestra["descripci_n_del_procedimiento"]
y = muestra["modalidad_de_contratacion"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Para aprender:", len(X_train), "| Para evaluar:", len(X_test))


In [ ]:
modelo = make_pipeline(
    TfidfVectorizer(max_features=1500, min_df=2),
    LogisticRegression(max_iter=500)
)
modelo.fit(X_train, y_train)             # aprende con textos de entrenamiento
prediccion = modelo.predict(X_test)      # infiere sobre textos no vistos
f1 = f1_score(y_test, prediccion, average="macro")
print("F1 macro observado:", round(f1, 4))


### 7A · Comparar con algo ingenuo

El clasificador de control siempre predice la clase mayoritaria. Si nuestro modelo no lo mejora, la complejidad no está justificada. F1 macro calcula F1 por clase y promedia sin ponderar por su frecuencia.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

control = DummyClassifier(strategy="most_frequent")
control.fit(X_train.to_frame(), y_train)
pred_control = control.predict(X_test.to_frame())
print("F1 macro control:", round(f1_score(y_test, pred_control, average="macro"), 4))
print("F1 macro modelo:", round(f1, 4))


**Cómo se lee:** F1 macro entre 0 y 1; comparar con el control usando el mismo test.  
**Qué nos dice:** rendimiento de un modelo sencillo para esta división de la muestra.  
**Qué NO permite concluir:** que funcione en otras entidades, periodos o modalidades; podría haber frases que revelan la etiqueta.  
**Error común:** ajustar TF-IDF antes de separar entrenamiento/prueba, o elegir parámetros observando reiteradamente el test.


## 8 · Dask Distributed: distribuir experimentos, no un único `fit`

Esta operación sí crea **un scheduler y dos workers lógicos** en Codespaces. Cada worker ejecuta una configuración independiente del modelo; dentro de cada tarea se usa scikit-learn normal.

**No es entrenamiento distribuido de un único modelo.** Para eso se necesita un algoritmo con soporte explícito, por ejemplo XGBoost Dask o Spark MLlib.


In [ ]:
from dask.distributed import Client
from sklearn.model_selection import cross_val_score

def probar(C, textos, etiquetas):
    candidato = make_pipeline(
        TfidfVectorizer(max_features=1500, min_df=2),
        LogisticRegression(C=C, max_iter=500)
    )
    resultados_cv = cross_val_score(
        candidato, textos, etiquetas, cv=3, scoring="f1_macro", n_jobs=1
    )
    return float(resultados_cv.mean())

with Client(processes=False, n_workers=2, threads_per_worker=1,
            dashboard_address=":8787") as cliente:
    print("Workers:", len(cliente.scheduler_info()["workers"]))
    print("Dashboard:", cliente.dashboard_link)
    valores_C = [0.1, 1.0, 10.0]
    textos_futuro = cliente.scatter(X_train, broadcast=True)
    etiquetas_futuro = cliente.scatter(y_train, broadcast=True)
    futuros = [
        cliente.submit(probar, C, textos_futuro, etiquetas_futuro, pure=False)
        for C in valores_C
    ]
    resultados_C = cliente.gather(futuros)

display(pd.DataFrame({"C": valores_C, "F1 CV": resultados_C}))


### 8A · ¿Para qué sirve el tablero Dask?

En Codespaces abre la pestaña **PORTS**, localiza **8787** y verifica que su visibilidad sea **Private**; abre la URL privada. En la vista de tareas observarás etapas de procesamiento y trabajadores. Si el puerto no se muestra, el resultado textual de `scheduler_info()` es suficiente.

**Cómo se lee:** cada tarea es un entrenamiento independiente.  
**Qué nos dice:** hay coordinación real con Dask dentro del Codespace.  
**Qué NO permite concluir:** que existan dos máquinas físicas o que el algoritmo `LogisticRegression.fit` esté distribuido.  
**Error común:** hablar de «entrenamiento distribuido» cuando sólo se paralelizan experimentos.


## 9 · Un error real, pero controlado

No vamos a matar un servidor ni borrar datos. Una tarea recibe entrada inválida, genera una excepción y después se ejecuta otra tarea correcta. Esto enseña diferencia entre **falla de tarea** y **caída de worker**.


In [ ]:
def duplicar(numero):
    if numero < 0:
        raise ValueError("Número inválido para el ejercicio")
    return numero * 2

with Client(processes=False, n_workers=2, threads_per_worker=1,
            dashboard_address=":8787") as cliente:
    try:
        cliente.submit(duplicar, -1, pure=False).result()
    except ValueError as error:
        print("Falló como esperábamos:", error)
    correcto = cliente.submit(duplicar, 4, pure=False).result()

print("Segundo intento controlado:", correcto)
assert correcto == 8


**Cómo se lee:** una tarea falla con entrada negativa y otra termina correctamente.  
**Qué nos dice:** podemos capturar fallos y decidir qué reintentar.  
**Qué NO permite concluir:** que Dask recupere automáticamente cualquier error lógico o garantice exactamente una escritura externa.  
**Error común:** confundir reintentar una tarea fallida con idempotencia de todo el sistema.


## 10 · Guardar evidencia reutilizable con Parquet

Parquet es un formato **columnar**. El archivo siguiente mantiene procesos únicos, modalidad y descripción; no guarda por sí mismo los pesos de un modelo entrenado. La sesión 14 lo leerá con PySpark.


In [ ]:
from analisis import escribir_resultados
manifest = escribir_resultados(tabla)
print("Salida:", manifest["salida"])
print("Filas:", manifest["filas"])
print("SHA-256 fuente:", manifest["sha256_fuente"][:20], "...")

archivo = RAIZ / "Laboratorios/S13_Codespaces/salidas/secop_s13.parquet"
reabierto = pd.read_parquet(archivo)
assert reabierto["id_del_proceso"].is_unique
print("Reabierto:", reabierto.shape)


**Cómo se lee:** se exportaron 1.000 procesos únicos de la muestra y se recuperaron desde Parquet.  
**Qué nos dice:** la entrega es tabular, trazable a una fuente específica y legible.  
**Qué NO permite concluir:** que el archivo Parquet tenga los modelos entrenados o preserve una licencia externa no registrada.  
**Error común:** confundir igualdad lógica de filas con igualdad byte a byte de dos archivos Parquet reescritos.


## 11 · Transferencia: convierte la técnica en tu propio proyecto

No memorices rutas del curso. En un proyecto nuevo identifica:

- **Archivo de entrada**: CSV o Parquet que controles y tengas permiso de usar.
- **Clave única**: ID que representa una observación.
- **Columna de agrupación**: qué necesitas comparar y por qué.
- **Etiqueta**: una variable que **exista realmente** si vas a entrenar supervisadamente.
- **Texto o variables**: de dónde provienen; ¿alguna revela directamente la etiqueta?
- **Control simple**: pandas antes de Dask.
- **Salida**: Parquet + manifiesto con fuente, conteos y decisiones.

En `Laboratorios/S13_Codespaces/analisis.py` puedes reemplazar `CSV`, `COL_TEXTO`, `COL_LABEL` y `CLASES` después de inspeccionar el nuevo esquema. Si la columna no existe, debes recibir un error visible, no fabricar valores.


## 12 · Autocomprobación técnica

Elige una herramienta para cada tarea **después de observar los resultados**, no sólo por definición:

| Caso | Primera respuesta defendible |
|---|---|
| 200 filas para un conteo | pandas |
| SQL sobre archivos grandes | considerar DuckDB |
| Varias transformaciones y tareas independientes | Dask según volumen y medición |
| Un modelo clásico pequeño | scikit-learn |
| Muchos entrenamientos de parámetros independientes | Dask Futures si la carga justifica coordinación |
| Un único modelo aprendido entre nodos | estimador con soporte distribuido explícito |

**Para llevar:** *procesar datos por particiones* ≠ *ejecutar varios modelos* ≠ *entrenar un modelo único de forma distribuida*.


## Laboratorio central · TF-IDF, clasificación y Dask con el mismo SECOP

**Ruta de clase:** estaciones 1–5 se realizan sobre una muestra pequeña; estación 6 consulta la evidencia medida con 50.000 procesos; estaciones 7–8 comparan arquitecturas y EmbeddingGemma; estación 9 completa la decisión estructurada. La estación 10 es una profundización **opcional**, no una tarea obligatoria.

**Regla de costo:** el cuaderno ejecuta una muestra pequeña; las pruebas de 50.000 registros, cuatro workers, modelos de 26.000 registros y generación de 1.000 embeddings permanecen desactivadas. Sus tiempos históricos deben leerse con el contexto y hardware indicados. **No actives ninguna prueba grande para completar el reto.**

**Dos recursos para aprender:** esta libreta y las diapositivas con la calculadora interactiva. El panel Streamlit es una herramienta en Codespaces, no una tercera lectura.



## 13B · Laboratorio ampliado: de un texto a una clasificación UNSPSC verificable

**Misión de Laura (Compras Claras):** recibir fichas SECOP II y proponer un segmento UNSPSC a partir del **título y la descripción**. El producto final es una matriz de texto, un clasificador evaluado, un benchmark local/Dask, una comparación real con EmbeddingGemma 2 y un informe reproducible. No se predice fraude ni un precio justo.

**Cinco estaciones obligatorias:** (1) entender TF, DF, IDF y normalización; (2) inspeccionar datos reales y las etiquetas; (3) entrenar un clasificador sencillo y comparar con uno ingenuo; (4) realizar *exactamente el mismo trabajo* local y con Dask; (5) evaluar cuándo compensa usar embeddings.

**Por qué este orden:** no podemos discutir paralelismo si no sabemos qué cálculo se reparte, ni F1 si desconocemos cómo se construye la variable objetivo. La presentación contiene una calculadora interactiva y el Codespace un panel de cinco actividades, pero tú también construirás cada parte aquí.

**Glosario:** *token* = unidad de texto; *vocabulario* = conjunto de términos considerados; *vector disperso* = conjunto de pesos donde la mayoría son cero; *fold* = partición de validación cruzada; *worker* = proceso que ejecuta tareas.


### Estación 1 · ¿Qué significa TF-IDF y por qué sirve?

**TF (Term Frequency, frecuencia de término)** cuenta cuántas veces aparece la palabra `t` en un documento `d`.

**DF (Document Frequency, frecuencia documental)** cuenta cuántos documentos distintos contienen `t`.

**IDF (Inverse Document Frequency, frecuencia documental inversa)** reduce la importancia de términos demasiado comunes. Scikit-learn usa la versión suavizada:

\[
\mathrm{IDF}(t) = \ln\left(\frac{1+N}{1+DF(t)}\right)+1
\]

El peso inicial es `TF × IDF`. Después se normaliza cada vector con la **norma L2**: su longitud euclídea se vuelve 1.

**Ejemplo para razonar antes de ejecutar:** A = «mantenimiento computadores»; B = «hospital computadores»; C = «hospital mantenimiento». Hay 3 documentos, cada término aparece en 2: DF = 2 e IDF = ln(4/3)+1 ≈ 1,28768. Si repites «computadores» en el documento A, su TF sube de 1 a 2, pero DF permanece igual.


In [ ]:
import math
from collections import Counter
import pandas as pd

mini_documentos = [
    "mantenimiento computadores",
    "hospital computadores",
    "hospital mantenimiento",
]
tokens = [doc.lower().split() for doc in mini_documentos]
vocabulario = sorted(set(sum(tokens, [])))
N = len(tokens)
filas = []
for termino in vocabulario:
    df = sum(termino in doc for doc in tokens)
    idf = math.log((1 + N) / (1 + df)) + 1
    for num, doc in enumerate(tokens):
        tf = doc.count(termino)
        filas.append({
            "documento": "ABC"[num], "termino": termino,
            "TF": tf, "DF": df, "IDF": round(idf, 5),
            "TF×IDF_sin_normalizar": round(tf * idf, 5),
        })
pesos_manuales = pd.DataFrame(filas)
display(pesos_manuales)
assert set(pesos_manuales["DF"]) == {2}
assert all(abs(v - (math.log(4/3)+1)) < 1e-4
           for v in pesos_manuales["IDF"])


**Cómo se lee:** una fila representa la palabra de un documento; TF se cuenta por fila, DF cuenta todos los documentos; IDF es común a todos los documentos del entrenamiento.  
**Qué nos dice:** los términos presentes en todos los documentos son menos informativos para diferenciarlos.  
**Qué NO permite concluir todavía:** que un proceso pertenece al segmento 80, ni que dos palabras sean sinónimas.  
**Error común:** confundir DF con número de veces que aparece una palabra dentro de un único documento.

**Pregunta para cambiar código:** añade «computadores» al documento A. ¿Qué columnas cambiarán? Predícelo antes de volver a ejecutar.


### Estación 2 · Contrastar la fórmula manual con scikit-learn

`TfidfVectorizer` primero construye un **vocabulario**, luego calcula la frecuencia documental, IDF, TF ponderado y normaliza cada fila. Usaremos `min_df=1` sólo para el ejemplo de tres documentos. En el benchmark grande `min_df=2` es razonable porque hay suficientes documentos.

**Mini ficha de funciones:**

| Función | Parámetros y devolución | Error frecuente |
|---|---|---|
| `fit()` | Aprende vocabulario e IDF del corpus de entrenamiento | Aplicarla sobre el test |
| `transform()` | Devuelve matriz dispersa de `filas × términos` | Interpretar un cero como dato faltante |
| `fit_transform()` | Hace ambas operaciones en entrenamiento | Repetirla sobre el test, cambiando columnas |
| `get_feature_names_out()` | Nombres de términos en orden de columnas | Suponer que cada vector es una etiqueta |


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

toy = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    min_df=1,
    norm="l2", smooth_idf=True, sublinear_tf=False,
)
mini_X = toy.fit_transform(mini_documentos)
mini_tabla = pd.DataFrame(
    np.round(mini_X.toarray(), 5),
    index=["A", "B", "C"],
    columns=toy.get_feature_names_out(),
)
display(mini_tabla)
assert mini_X.shape == (3, 3)
assert np.allclose(np.linalg.norm(mini_X.toarray(), axis=1), 1.0)


**Cómo se lee:** cada fila es el vector de un texto; cada columna es una palabra. Los valores ya están normalizados, por eso no son iguales a los productos TF×IDF sin normalizar de la celda anterior.  
**Qué nos dice:** podemos reproducir la aritmética y comprobar matemáticamente lo que entrega la biblioteca.  
**Qué NO permite concluir todavía:** que el algoritmo entiende contexto semántico o calidad del contrato.  
**Error común:** creer que los pesos de TF-IDF son probabilidades de cada categoría.

**Transferencia:** prueba `norm=None`, compara la matriz y explica por qué cambió.


### Estación 3 · Similitud coseno: buscar un expediente sin entrenar un clasificador

TF-IDF permite **recuperar textos relacionados** incluso sin etiquetas. La consulta y los documentos deben representarse con el mismo vocabulario. La similitud coseno mide su alineación:

\[
\cos(\theta) = \frac{a\cdot b}{\|a\|\,\|b\|}
\]

Con vectores normalizados L2, equivale al producto punto. En un corpus de TF-IDF no negativo suele estar entre 0 y 1; un 0 puede significar palabras distintas, no temas necesariamente diferentes.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

consulta = "quiero mantenimiento de computadores"
consulta_vector = toy.transform([consulta])
similares = cosine_similarity(consulta_vector, mini_X).ravel()
tabla_similitud = pd.DataFrame({
    "documento": ["A", "B", "C"],
    "texto": mini_documentos,
    "similitud_coseno": np.round(similares, 4),
}).sort_values("similitud_coseno", ascending=False)
display(tabla_similitud)
assert len(similares) == 3


**Cómo se lee:** el documento con coseno más alto comparte términos relevantes con la consulta.  
**Qué nos dice:** TF-IDF habilita una búsqueda léxica reproducible sin llamadas a modelos grandes.  
**Qué NO permite concluir todavía:** que frases con sinónimos no presentes sean irrelevantes.  
**Error común:** construir otro `TfidfVectorizer` para la consulta; los índices de columnas ya no coincidirían.

**Reto:** reemplaza «computadores» por «PC» y observa si la similitud cae. Esa limitación motiva probar EmbeddingGemma 2.


### Estación 4 · Inspeccionar SECOP real antes de usarlo

**Fuente:** `Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv` contiene 1.000 fichas SECOP y varios campos administrativos. Para clasificar usamos el código `V1.80111600` como segmento `80`, no el prefijo `V1`. Un `UNSPECIFIED` no se inventa. Tampoco usaremos `precio_base=0` o el contador de ofertas como objetivos.

**Funciones nuevas:** `pd.read_csv(usecols=...)` lee sólo las columnas necesarias; `drop_duplicates()` impide repetir IDs; `re.fullmatch()` garantiza el formato completo; `pd.to_datetime(errors="coerce")` marca fechas inválidas sin atribuirles un valor falso.


In [ ]:
import re
from pathlib import Path

archivo_secop = RAIZ / "Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv"
columnas_secop = [
    "id_del_proceso", "nombre_del_procedimiento",
    "descripci_n_del_procedimiento", "codigo_principal_de_categoria",
    "fecha_de_publicacion_del",
]
secop_pequeno = pd.read_csv(
    archivo_secop, usecols=columnas_secop,
    dtype=str, keep_default_na=False,
).drop_duplicates("id_del_proceso")
patron = re.compile(r"^(?:V[0-9]+\.)?([0-9]{8})$")
def segmento_unspsc(codigo):
    m = patron.fullmatch(str(codigo).strip())
    return m.group(1)[:2] if m else None

secop_pequeno["segmento"] = secop_pequeno[
    "codigo_principal_de_categoria"].map(segmento_unspsc)
secop_pequeno["texto"] = (
    secop_pequeno["nombre_del_procedimiento"] + " " +
    secop_pequeno["descripci_n_del_procedimiento"]
).str.strip()
print("Procesos únicos:", len(secop_pequeno))
print("Sin segmento válido:", secop_pequeno["segmento"].isna().sum())
display(secop_pequeno[["id_del_proceso", "segmento", "texto"]].head(5))


**Cómo se lee:** cada fila es una ficha de un proceso real; la columna segmento contiene el primer par de dígitos del código UNSPSC válido.  
**Qué nos dice:** podemos crear etiquetas administrativas rastreables sin adivinar ceros.  
**Qué NO permite concluir todavía:** que las categorías registradas estén libres de errores.  
**Error común:** convertir `V1.80111600` quitando caracteres no numéricos y tomar «18» como primer segmento; eso mezcla la versión con el código.

**Reto:** identifica cuántos códigos faltan y cuántos pertenecen al segmento 80; reflexiona sobre el desbalance.


### Estación 5 · Entrenar un modelo que sí podemos explicar: TF-IDF + LinearSVC

`LinearSVC` encuentra límites de decisión entre las categorías representadas por TF-IDF. Es un clasificador lineal: no genera embeddings neuronales. En este caso de seis categorías, **F1 macro** promedia la capacidad de predecir correctamente cada categoría por igual.

Antes de evaluar, ordenamos por fecha y reservamos los procesos más recientes. Ajustaremos TF-IDF **únicamente con train**. Comparamos contra el control que siempre predice la clase frecuente. En las muestras grandes, el benchmark registra la huella del test fijo y emplea los mismos hiperparámetros en todos los tamaños.


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline

clases_objetivo = {"80", "85", "93", "81", "72"}
ejemplos = secop_pequeno[
    secop_pequeno["segmento"].notna()
    & secop_pequeno["texto"].str.len().ge(20)
].copy()
ejemplos["clase"] = ejemplos["segmento"].where(
    ejemplos["segmento"].isin(clases_objetivo), "OTRAS")
ejemplos["fecha"] = pd.to_datetime(
    ejemplos["fecha_de_publicacion_del"], errors="coerce")
ejemplos = ejemplos.dropna(subset=["fecha"]).sort_values(
    ["fecha", "id_del_proceso"])
i = int(.8 * len(ejemplos))
train_secop, test_secop = ejemplos.iloc[:i], ejemplos.iloc[i:]
modelito = make_pipeline(
    TfidfVectorizer(max_features=3000, min_df=2, ngram_range=(1,2)),
    LinearSVC(class_weight="balanced", random_state=42),
)
modelito.fit(train_secop["texto"], train_secop["clase"])
predicciones = modelito.predict(test_secop["texto"])
control = DummyClassifier(strategy="most_frequent")
control.fit(np.zeros((len(train_secop),1)), train_secop["clase"])
base_pred = control.predict(np.zeros((len(test_secop),1)))
etiquetas_fijas = ["72","80","81","85","93","OTRAS"]
print("Train:",len(train_secop),"Test:",len(test_secop))
print("F1 macro modelo:", round(f1_score(
    test_secop["clase"], predicciones,
    labels=etiquetas_fijas, average="macro", zero_division=0),4))
print("F1 macro mayoritaria:", round(f1_score(
    test_secop["clase"], base_pred,
    labels=etiquetas_fijas, average="macro", zero_division=0),4))


**Cómo se lee:** F1 macro cercano a 1 sería mejor; en este conjunto pequeño importa compararlo con el control y revisar la F1 de las categorías minoritarias.  
**Qué nos dice:** si el texto aporta señales para aproximar la etiqueta administrativa.  
**Qué NO permite concluir todavía:** que el sistema generalice a SECOP completo ni que esté validado para hacer revisiones jurídicas.  
**Error común:** reportar sólo exactitud general cuando el segmento 80 domina.

**Reto:** ajusta sólo `ngram_range=(1,1)` y compara la calidad sobre el **mismo test**. No ajustes los hiperparámetros observando repetidamente el test final.


### Estación 6 · Comparación justa en 1.000, 10.000, 20.000 y 50.000 procesos

Ahora separamos **10.000 procesos fijos de prueba** y mantenemos los mismos IDs, clases, modelo e hiperparámetros. Los cuatro entrenamientos están anidados en un mismo conjunto de 50.000 procesos más antiguos. De otro modo, una F1 distinta podría provenir de un test distinto, no del tamaño del entrenamiento.

Los resultados reales de GitHub Actions fueron:

| Entrenamiento | TF-IDF local, transformación | TF-IDF Dask, transformación | F1 macro LinearSVC sobre 10k test |
|---:|---:|---:|---:|
| 1.000 | 0,5773 s | 0,4028 s | 0,48914 |
| 10.000 | 1,0469 s | 0,6435 s | 0,55635 |
| 20.000 | 1,5667 s | 0,9170 s | 0,56585 |
| 50.000 | 3,0996 s | 1,7655 s | 0,57385 |

Son medianas de tres transformaciones sobre **train + test**, excluyen el tiempo de leer CSV, crear el clúster, ajustar el vocabulario y entrenar el modelo. **No extrapoles esta ganancia al tiempo total**.

**Contraste con el nuevo XGBoost (solo entrenamiento, comparación corregida):** en otra ejecución real sobre el mismo protocolo temporal, con 50.000 fichas de train y 10.000 de test, XGBoost local (dos hilos) tardó **0,5816 s** y XGBoost Dask (dos workers de un hilo) **1,2465 s**. Esos tiempos se miden después de preparar las matrices, sin incluir predicción ni creación del clúster; los resultados históricos anteriores que sí incluían predicción local **no son comparables**. [Ver ejecución y QA del benchmark corregido](https://github.com/jazaineam1/BigData2026/actions/runs/43175326562).

**Cómo se lee:** son dos entrenamientos equivalentes en el mismo runner.  
**Qué nos dice:** distribuir este entrenamiento pequeño añadió costo en la máquina medida.  
**Qué NO permite concluir:** que Dask siempre pierde, ni que los segundos sean idénticos en tu Codespace.  
**Error común:** sumar o comparar estos tiempos con los de TF-IDF como si fueran el mismo cronómetro.


In [ ]:
# Ejecuta sólo si deseas medir tú mismo los cuatro tamaños en Codespaces.
import subprocess
import sys

EJECUTAR_50K_AHORA = False
benchmark_justo = RAIZ / "Laboratorios/S13_Codespaces/benchmark_50k_test_fijo.py"
assert benchmark_justo.exists()
if EJECUTAR_50K_AHORA:
    subprocess.run(
        [sys.executable, str(benchmark_justo),
         "--sizes","1000","10000","20000","50000",
         "--workers","2","--repeticiones","3","--rondas","20"],
        cwd=RAIZ, check=True)
else:
    print("Extensión de 50k opcional; activa EJECUTAR_50K_AHORA.")


### Estación 7 · Modelo idéntico, tareas distribuidas: tres folds en paralelo

**Problema más sencillo donde Dask puede ayudar:** la validación cruzada requiere entrenar **el mismo TF-IDF + LinearSVC varias veces**. Cada fold tiene datos de train y validación distintos, pero la función, sus parámetros y la semilla son exactamente los mismos.

- **Secuencial:** tres entrenamientos uno detrás de otro.
- **joblib, dos procesos:** los mismos tres entrenamientos paralelos en la máquina local.
- **Dask, dos workers:** los mismos tres entrenamientos a través de un scheduler y workers.
- **Condición de validez:** las tres F1 por fold **deben coincidir**, aunque los tiempos no.

No afirmamos que un único LinearSVC entrene distribuido. Aquí Dask distribuye **entrenamientos independientes**, lo que es una capacidad muy útil en búsqueda de hiperparámetros, pruebas y validación cruzada.


In [ ]:
# Lee código abierto de la prueba CV y observa exactamente la tarea de un fold.
import inspect
import sys
laboratorio = RAIZ / "Laboratorios/S13_Codespaces"
if str(laboratorio) not in sys.path:
    sys.path.insert(0,str(laboratorio))
from cv_dask_mismo_modelo import entrenar_un_fold
print(inspect.getsource(entrenar_un_fold))
# Dentro de Codespaces:
# python Laboratorios/S13_Codespaces/cv_dask_mismo_modelo.py --sizes 1000 10000 20000 50000 --cv 3 --workers 2


**Cómo se lee:** la función que acabas de imprimir no cambia entre la ejecución local, la multiproceso y Dask.  
**Qué nos dice:** si Dask gana, lo hizo por coordinación de tareas independientes, no por un algoritmo más preciso.  
**Qué NO permite concluir todavía:** que en un servidor con menos CPU o más red Dask mantenga la ventaja.  
**Error común:** comparar Dask 4 workers contra un único proceso local y adjudicar toda la mejora al framework, cuando también cambió la cantidad de CPU.

En Codespaces puedes escoger 2 o 4 CPU según tu disponibilidad. Si tienes 4 CPU, compara local 2 procesos y Dask 2 workers con la misma cantidad de hilos, y **documenta el costo de los recursos**.


### Estación 8 · EmbeddingGemma 2: 1.000 documentos probados de verdad

En vez de reentrenar los pesos de EmbeddingGemma, pedimos a Ollama el vector de **768 dimensiones** de cada texto y entrenamos el **mismo LinearSVC** con las mismas etiquetas UNSPSC y el mismo corte 800/200 de la comparación original.

Resultados medidos en GitHub Actions:

| | TF-IDF | EmbeddingGemma 2 (270m) |
|---|---:|---:|
| F1 macro | 0,45134 | 0,47586 |
| Balanced accuracy | 0,43305 | 0,50070 |
| Vectorización de 1.000 textos | 0,111 s | 119,898 s |
| Dimensiones | 4.201 | 768 |

**Cómo se lee:** el modelo neuronal mejoró F1 macro en 0,02452 puntos absolutos, pero la generación de vectores fue aproximadamente mil veces más costosa en ese runner.  
**Qué nos dice:** debemos decidir si esa mejora compensa el tiempo de inferencia y el mantenimiento de los pesos.  
**Qué NO permite concluir todavía:** que EmbeddingGemma mejore 10k/20k/50k, ni su rendimiento en el HP.  
**Error común:** decir «se entrenó EmbeddingGemma». Aquí sólo generamos sus representaciones y entrenamos LinearSVC encima.


In [ ]:
# El comando crea 1000/1000 embeddings de verdad cuando Ollama está activo.
# No se activa en Run All porque descarga un modelo y requiere un servidor Ollama.
EJECUTAR_1000_OLLAMA = False

script_gemma = RAIZ / "Laboratorios/S13_Codespaces/comparar_1000_embeddings_reales.py"
assert script_gemma.exists()
if EJECUTAR_1000_OLLAMA:
    subprocess.run(
        [sys.executable, str(script_gemma), "--batch", "8"],
        cwd=RAIZ, check=True)
else:
    print("Prueba de 1000 EmbeddingGemma preparada. Requiere Ollama local en Codespaces o HP.")


### Estación 9 · Una herramienta que los estudiantes pueden modificar

Con el Codespace S13 abierto (Code → Codespaces → **New with options → S13**), desde la raíz del repositorio:

```bash
python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt
streamlit run Laboratorios/S13_Codespaces/app_tfidf_unspsc.py --server.address 0.0.0.0 --server.port 8503
```

Abre **PORTS → 8503 → Open in Browser** y conserva la visibilidad en **PRIVATE**. Cinco pestañas muestran datos SECOP, cálculos manuales de TF-IDF, búsqueda coseno, clasificación y métricas Dask.

**Reto de transferencia (práctica formativa, no nueva calificación):** modifica una palabra del corpus, cambia la consulta y `ngram_range` sin tocar el test; registra las salidas reales de DF/IDF, F1 macro y tiempos local/Dask. Entrega el código ejecutado, dos gráficos y tabla de clases. Para decidir y declarar límites, selecciona alternativas estructuradas basadas en tus propias mediciones, sin respuestas abiertas.

**Rúbrica observable de práctica (10 puntos, retroalimentación formativa):**

| Criterio verificable | 2 puntos | 1 punto | 0 puntos |
|---|---|---|---|
| Ejecución y traza | Código ejecutable y enlace a commit propio | Código ejecutable sin commit | Sin ejecución verificable |
| TF, DF e IDF | Tabla antes/después con valores reproducibles | Solo una tabla reproducible | Sin valores reproducibles |
| Evaluación de clasificación | F1 macro y tabla por clase sobre el mismo test | Solo F1 macro sobre el test | Cambia el test o no reporta métricas |
| Comparación de rendimiento | Local y Dask con misma operación, tiempos y número de workers | Tiempos sin equivalencia o recursos | Sin medición |
| Interpretación de límites | Registra una elección numérica de arquitectura y un límite específico usando la celda de selección que sigue | Selección sin evidencia propia o límite | Atribuye causalidad/fraude o no selecciona |

**Nota de evaluación:** esta rúbrica no crea por sí misma una nota oficial ni modifica los porcentajes del PDA; cualquier calificación formal exige vincular el ejercicio con el componente vigente del PDA.

**Producto a S14:** datasets Parquet y manifiesto de procedencia; podremos leer los mismos procesos con PySpark sin cambiar su interpretación.

**Evidencia verificable:** ejecuta la celda de selección inmediatamente posterior a esta estación; generará `decision_s13.json` solo si completas ambas decisiones. **No hay respuesta correcta preseleccionada.** En la decisión usa tus propios tiempos y considera el costo de arranque del clúster.


In [ ]:
# Reto de cierre · respuesta estructurada y verificable (sin texto libre)
# Modifica SOLO los dos números según TUS mediciones. Ninguno está preseleccionado.
# ARQUITECTURA: 1 = pandas/TF-IDF local; 2 = Dask TF-IDF 2 workers;
# 3 = no hay evidencia suficiente para cambiar arquitectura.
# LIMITE: 1 = no contamos arranque del clúster; 2 = muestra no representativa;
# 3 = etiquetas UNSPSC administrativas y desequilibradas;
# 4 = los tiempos observados dependen del hardware.
ARQUITECTURA = None
LIMITE = None

OPCIONES_ARQUITECTURA = {
    1: "Continuar con TF-IDF local",
    2: "Distribuir la transformación TF-IDF con Dask (2 workers)",
    3: "No decidir aún: medir costo extremo a extremo",
}
OPCIONES_LIMITE = {
    1: "El cronómetro de transformación excluye iniciar el clúster",
    2: "La muestra evaluada no representa todos los procesos SECOP",
    3: "UNSPSC es una etiqueta administrativa y las clases son desiguales",
    4: "La comparación se hizo en un hardware concreto",
}
print("Elige UNA arquitectura y UN límite de los siguientes menús:")
print("Arquitectura:", OPCIONES_ARQUITECTURA)
print("Límite:", OPCIONES_LIMITE)
if ARQUITECTURA is None or LIMITE is None:
    print("Práctica pendiente: cambia ambos None por los índices seleccionados y vuelve a ejecutar.")
else:
    if type(ARQUITECTURA) is not int or ARQUITECTURA not in OPCIONES_ARQUITECTURA:
        raise ValueError("ARQUITECTURA debe ser exactamente 1, 2 o 3")
    if type(LIMITE) is not int or LIMITE not in OPCIONES_LIMITE:
        raise ValueError("LIMITE debe ser exactamente 1, 2, 3 o 4")
    import json
    from pathlib import Path
    evidencia = {
        "sesion": 13,
        "arquitectura_codigo": ARQUITECTURA,
        "arquitectura": OPCIONES_ARQUITECTURA[ARQUITECTURA],
        "limite_codigo": LIMITE,
        "limite": OPCIONES_LIMITE[LIMITE],
        "fuente": "mediciones propias del cuaderno / resultados históricos diferenciados"
    }
    salida_decision = RAIZ / "Laboratorios/S13_Codespaces/salidas/decision_s13.json"
    salida_decision.parent.mkdir(parents=True, exist_ok=True)
    salida_decision.write_text(json.dumps(evidencia,ensure_ascii=False,indent=2), encoding="utf-8")
    print("Evidencia exportada a:", salida_decision)
    print("Comprueba que la arquitectura seleccionada corresponda a tus tiempos, no a un resultado supuesto.")


### Estación 10 · cuatro CPU y el mismo TF-IDF: la evidencia supera la intuición

Laura repite la transformación de **50.000 fichas de entrenamiento y 10.000 de prueba** en una máquina con cuatro CPU lógicas. La misma representación y los mismos parámetros se prueban tres veces, con scikit-learn local, Dask 2 workers y Dask 4 workers. **Los vectores producidos son idénticos**.

| Método | Mediana de tres transformaciones | Arranque del clúster |
|---|---:|---:|
| Local | 1,8497 s | 0 |
| Dask 2 | 1,1595 s | 1,3460 s |
| Dask 4 | 1,0259 s | 1,4587 s |

**Cómo se lee:** menor tiempo de transformación es mejor; el arranque es un costo adicional distinto.  
**Qué nos dice:** pasar de 2 a 4 workers ahorró 0,1336 segundos en esta operación, mucho menos que el costo de crear el clúster.  
**Qué NO permite concluir:** que la inferencia completa de 50.000 contratos sea más rápida, o que Dask funcione más rápido en cualquier equipo con cuatro CPU. Aquí hay una sola VM y sólo se mide transformar el texto.  
**Error común:** comparar distintos clasificadores o distintos conjuntos de prueba y atribuir el tiempo a la distribución.

**Decisión verificable, no respuesta abierta:** en la siguiente celda elige el tamaño de máquina que tienes, activa opcionalmente las mediciones y exporta el resultado JSON. Contrasta los tiempos con los propios y guarda el commit.


In [ ]:
# Medición opcional: NO se ejecuta durante Run All por tiempo/CPU.
import subprocess, sys, os
ACTIVAR_PRUEBA_4CPU = False
N_CPU = os.cpu_count() or 1
print(f"CPU lógicas disponibles: {N_CPU}")
if ACTIVAR_PRUEBA_4CPU:
    if N_CPU < 4:
        raise RuntimeError("Para comparar 4 workers cambia el Codespace a 4 CPU o más.")
    subprocess.run(
        [sys.executable, str(RAIZ / "Laboratorios/S13_Codespaces/medir_workers_50k.py"),
         "--reps", "3"], check=True, cwd=RAIZ)
else:
    print("Prueba no activada; consulta el JSON real del CI y decide si tu máquina tiene recursos.")


## Cierre · Puente a la sesión 14

El resultado de la sesión 13 es `secop_s13.parquet`, un CSV fuente identificado, un modelo clásico evaluado y evidencias de coordinación. **En S14 usaremos PySpark para leer el mismo conjunto y comparar un tercer motor de procesamiento**, no volver a empezar otro caso.

**Errores comunes que evitamos:** usar Dask para 200 filas sin justificarlo; contar modalidades como irregularidades; promediar promedios; llamar «clúster físico» a dos workers locales; ajustar TF-IDF antes del corte; confundir inferencia y entrenamiento; comparar tiempos sin mismo trabajo.

**Fuentes oficiales:** [Dask](https://docs.dask.org/en/stable/) · [Dask Distributed](https://distributed.dask.org/) · [scikit-learn](https://scikit-learn.org/stable/) · [Codespaces](https://docs.github.com/es/codespaces).


## Apéndice opcional · Experimentos históricos de regresión y ampliación técnica

Este bloque fue conservado para consulta profesional y **no es parte del recorrido central** de la sesión. La regresión exploratoria de precios **no estima fraude ni sobrecostos** y los valores cero no significan ausencia de competencia. No ejecutes los scripts costosos salvo una necesidad específica, con cuota y recursos comprobados.


**ADVERTENCIA DE REVISIÓN CIENTÍFICA:** la siguiente sección de regresión de precios se conserva como demostración técnica histórica de XGBoost distribuido, pero NO es el modelo recomendado; los ceros, diferencias entre modalidades y fuga de información impiden usar el residual como alerta contractual. Continúa con la sección final de clasificación UNSPSC sobre 20.000 procesos reales.\n\n> **AUDITORÍA DE CALIDAD (noviembre de 2026).** En 300.000 registros SECOP encontramos 11.392 precios base cero y 298.726 contadores de ofertas cero. El modelo NO utiliza el conteo de ofertas ni convierte sus ceros en observaciones de no competencia. El modelo de precio sirve sólo para **demostrar entrenamiento distribuido**, no para calcular precios justos ni producir alertas contractuales. También se corrigió un error previo de UNSPSC: `V1.80111600` corresponde al segmento **80**, no **18**. **No utilices modelos anteriores a la corrección.** [Auditoría reproducible](../docs/S13_AUDITORIA_DATOS_SECOP_300K.md).

## Extensión: 26.000 procesos SECOP y un único XGBoost distribuido

**Caso Compras Claras:** Laura mide qué tan bien un modelo reproduce el precio base registrado. Sus errores **NO identifican contratos caros o baratos** y no se usarán para generar alertas.

**¿Por qué cambiamos de modelo?** En la sección anterior, Dask repartía entrenamientos independientes de scikit-learn. Ahora distribuiremos **las filas que necesita un único XGBoost** mediante `xgboost.dask.DaskDMatrix` y `xgboost.dask.train()`.

La fuente ya está en el repositorio: tres CSV de procesos SECOP II. Se muestrean de manera reproducible **26.000 fichas reales con ID único** tras revisar todos los CSV, se entrenan **20.800** y se evalúan **5.200** más recientes. *No son 26.000 archivos PDF*.

[Código del motor editable](../Laboratorios/S13_Codespaces/modelo_distribuido_20k.py) · [Guía técnica de instalación, datos y métricas](../Laboratorios/S13_Codespaces/GUIA_MODELO_20K.md).


### 1 · Elegir variables antes de elegir modelos

| Sistema | Entradas | ¿Qué aprendizaje hay? |
|---|---|---|
| Control | mediana de precios de entrenamiento | no aprende relaciones |
| XGBoost A | modalidad, región, duración, tipo, categoría, mes | un modelo entrenado entre workers Dask |
| XGBoost B | igual + 16 ejes TF-IDF/SVD del texto | otro modelo distribuido para medir aporte del texto |
| EmbeddingGemma | embeddings neuronales del texto | **opcional futuro**, sólo si el costo y la mejora lo justifican |

**TF-IDF/SVD no son embeddings neuronales.** La variable objetivo es `log1p(precio_base)`; el logaritmo ayuda con montos de muy distinta escala.

**Fuga de información:** nunca entrenamos con `valor_total_adjudicacion`, `adjudicado` o respuestas posteriores. El texto podría mencionar precios o condiciones muy específicas; se debe examinar antes de desplegar.


### 2 · Código fundamental que debes entender

```python
dX = da.from_array(X_train, chunks=(2500, X_train.shape[1]))
dy = da.from_array(y_train, chunks=(2500,))
dtrain = xgboost.dask.DaskDMatrix(cliente, dX, dy, feature_names=nombres)
resultado = xgboost.dask.train(
    cliente,
    {"objective": "reg:squarederror", "tree_method": "hist"},
    dtrain, num_boost_round=30,
)
```

**Qué hace:** divide filas de entrenamiento en particiones, reparte los datos y coordina el cálculo de los árboles. El estimador XGBoost combina información entre workers para **un único modelo**, no sólo tres funciones `fit()` independientes.

**Dónde corre:** 2 workers con procesos separados en un Codespace remoto. Sigue siendo **una máquina física**. Un modelo local de 20.800 procesos podría ser más rápido; aquí investigamos cómo funciona el mecanismo distribuido.


### 3 · Ejecutar un entrenamiento real desde tu Codespace

El propio programa `modelo_distribuido_20k.py` permite leer y cambiar cada función, y expone los parámetros de tamaño, rondas y número de trabajadores. Prueba en la terminal:

```bash
python -m pip show xgboost
python Laboratorios/S13_Codespaces/modelo_distribuido_20k.py --registros 26000 --rondas 30 --workers 2
```

Si falta XGBoost, dentro del Codespace: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`.

Activa la celda siguiente poniendo `ACTIVAR_MODELO_26K=True`. Está desactivada por defecto para que **Run All del laboratorio básico** no repita una segunda ejecución costosa; la prueba independiente del proyecto sí realiza un entrenamiento real sobre los 26.000 procesos.


In [ ]:
import subprocess
import sys

ACTIVAR_MODELO_26K = False  # Cambiar a True para el entrenamiento real.
motor_ml = RAIZ / "Laboratorios/S13_Codespaces/modelo_distribuido_20k.py"
assert motor_ml.exists(), "Falta el motor de aprendizaje del repositorio"

if ACTIVAR_MODELO_26K:
    subprocess.run(
        [sys.executable, str(motor_ml), "--registros", "26000", "--rondas", "30"],
        cwd=RAIZ, check=True
    )
else:
    print("Extensión lista: pon ACTIVAR_MODELO_26K=True cuando quieras entrenar.")


### 4 · Ver resultados que existen de verdad

El entrenamiento produce un manifiesto de fuentes (SHA-256), dos modelos JSON, un Parquet con los 26.000 procesos, y predicciones sólo sobre la prueba temporal. Los ceros del precio no se imputan; su cantidad se registra en el manifiesto. Compara cada modelo con la **mediana del entrenamiento** usando el mismo periodo de prueba.

- **MAE(log)**: error en la escala logarítmica; menor es mejor.
- **Factor mediano**: diferencia multiplicativa típica; 1 es perfecto.
- **Dentro de ×2**: porcentaje de casos cuyo precio estimado está entre la mitad y el doble del observado.
- **Error multiplicativo ≥×3**: identifica dónde se equivocó el modelo en la prueba, **no un contrato que requiera una alerta**.

Para comprobar los resultados ejecutados desde la terminal, usa la siguiente celda.


In [ ]:
import json
from pathlib import Path
import pandas as pd

dir_modelo = RAIZ / "Laboratorios/S13_Codespaces/salidas/modelo_20k"
archivo_manifest = dir_modelo / "manifiesto.json"
if archivo_manifest.is_file():
    metadatos = json.loads(archivo_manifest.read_text(encoding="utf-8"))
    print("Fichas:", metadatos["observaciones_unicas"],
          "| entrenamiento:", metadatos["entrenamiento"],
          "| test:", metadatos["prueba"],
          "| workers:", metadatos["workers"])
    display(pd.DataFrame(metadatos["metricas"]).T)
else:
    print("No hay métricas 26K todavía: ejecuta el motor antes de interpretarlas.")


### 5 · Laboratorio de investigación visual

Después de ejecutar el motor, abre un panel interactivo en la **terminal del Codespace**:

```bash
streamlit run Laboratorios/S13_Codespaces/app_modelo_20k.py --server.address 0.0.0.0 --server.port 8502
```

Desde **PORTS**, abre el puerto **8502** con visibilidad **PRIVATE**. Explora errores de predicción por modalidad y magnitud de diferencia, mira la nube de precio real frente a previsto y abre la URL SECOP original para comprobar un caso.

**Cómo se lee:** compara factores y tres métricas entre modelos. **Qué nos dice:** dónde el modelo se equivoca más. **Qué NO permite concluir:** presupuesto justo, ilegalidad, daño fiscal o desviación causal. **Error común:** confundir una predicción con una auditoría jurídica.

**Puente a S14:** el archivo `muestra_26k.parquet` se puede leer con PySpark para comparar cómo cambiaría la ejecución en otro motor.


## Nuevo experimento principal · ¿TF-IDF o EmbeddingGemma 2 para clasificar SECOP?

**La pregunta correcta después de auditar los ceros:** ¿podemos sugerir la categoría UNSPSC declarada de un proceso a partir de su título y descripción? No se predice precio, fraude ni número de oferentes.

Se usaron los tres CSV SECOP originales (300.000 filas) y se seleccionaron **20.000 procesos únicos con código UNSPSC válido**. Hay seis clases: segmentos 80, 85, 93, 81 y 72, más «OTRAS». El código UNSPSC se enmascara cuando aparece como secuencia numérica larga en el texto.

Se comparan 1.000, 10.000 y 20.000 fichas, con 80 % de entrenamiento y 20 % de prueba ordenada temporalmente. Se comprueba que las transformaciones Dask y locales produzcan exactamente los mismos vectores TF-IDF, antes de hablar de velocidad.

**Resultados medidos en GitHub Actions** (4 CPU lógicas, dos workers en una sola máquina):

| Procesos | TF-IDF local (transformación) | TF-IDF Dask (transformación) | F1 macro TF-IDF/LinearSVC | F1 macro SVD/XGBoost Distributed |
|---:|---:|---:|---:|---:|
| 1.000 | 0,042 s | 0,115 s | **0,4513** | 0,3851 |
| 10.000 | 0,417 s | 0,329 s | **0,5594** | 0,3960 |
| 20.000 | 0,892 s | 0,688 s | **0,5653** | 0,4088 |

**No confundas** tiempo de `transform()` con tiempo de toda la tubería: no incluye vocabulario, lectura CSV, inicio del clúster ni entrenamiento. Crear el clúster costó 1,454 segundos. Los tamaños tienen pruebas temporales diferentes, por lo que la F1 creciente es descriptiva, no una curva de aprendizaje controlada con un test idéntico.

[Benchmark completo y resultados](../Laboratorios/S13_Codespaces/GUIA_BENCHMARK_TFIDF_EMBEDDINGGEMMA.md).


### Código abierto, no un programa mágico

El script `benchmark_unspsc_tfidf_dask.py` deja ver las funciones `leer_muestra`, `seleccion_n`, `transformar_dask` y `entrenar_xgb_distribuido`. **Cada función puede reemplazarse** y se comprueba si cambia la respuesta.

Ejecuta desde la terminal (o activa la celda opcional):

```bash
python Laboratorios/S13_Codespaces/benchmark_unspsc_tfidf_dask.py --sizes 1000 10000 20000 --workers 2 --rondas 20 --svd-dim 32
```

**Resultados:** tres JSON con métricas y un `benchmark.json` dentro de `salidas/benchmark_unspsc/`.

**Cómo se lee:** menor tiempo de transformación, mejor; mayor F1 macro, mejor.  
**Qué nos dice:** si el costo de Dask compensa en un tamaño dado y si el clasificador ayuda a distinguir categorías.  
**Qué NO permite concluir:** que el segmento declarado sea correcto, ni que Dask sea más rápido en un clúster distinto.  
**Error común:** interpretar una predicción «OTRAS» como un problema contractual.


In [ ]:
# Ejecución OPCIONAL. Se deja desactivada para no duplicar 3 pruebas en cada Run All.
import subprocess
import sys

ACTIVAR_BENCHMARK_TFIDF = False
script_bench = RAIZ / "Laboratorios/S13_Codespaces/benchmark_unspsc_tfidf_dask.py"
assert script_bench.exists()
if ACTIVAR_BENCHMARK_TFIDF:
    subprocess.run([
        sys.executable, str(script_bench),
        "--sizes", "1000", "10000", "20000",
        "--workers", "2", "--rondas", "20", "--svd-dim", "32"
    ], cwd=RAIZ, check=True)
else:
    print("Benchmark opcional listo; pasa ACTIVAR_BENCHMARK_TFIDF=True para medir.")


### EmbeddingGemma 2: usar sus vectores no significa entrenar sus pesos

`embeddinggemma-2:270m` es la versión sólo texto. Ollama produce un vector de 768 componentes para cada descripción; después entrenamos un **LinearSVC** con esos vectores y las mismas etiquetas UNSPSC. Los vectores se almacenan en SQLite para reanudar después de interrupciones.

```bash
ollama pull embeddinggemma-2:270m
python Laboratorios/S13_Codespaces/medir_embeddinggemma2.py --n 48 --lote 4 --rondas 2
python Laboratorios/S13_Codespaces/clasificar_unspsc_embeddinggemma2.py --n 1000 --batch 4
```

**Antes de creer una estimación de tiempo**: usa el primer comando Python para calcular los textos por segundo en TU equipo. No podemos prometer el rendimiento del HP a partir de tiempos en GitHub Actions.

**Cómo se lee:** compara F1 macro y segundos de generación de vectores; el primer uso incluye carga del modelo.  
**Qué nos dice:** si la semántica de EmbeddingGemma mejora categorías que TF-IDF confunde.  
**Qué NO permite concluir:** que entrena Gemma desde cero, ni que Dask duplica la capacidad de una CPU.  
**Error común:** configurar dos workers Dask contra un único servidor Ollama y asumir que habrá el doble de rendimiento.

**Reto:** comienza por 1.000 documentos; si EmbeddingGemma mejora la F1 de 0,4513, entonces calcula si el tiempo adicional para 10.000 o 20.000 se justifica. Repite sobre las mismas particiones temporales.
